# CNN Dog vs. Cat Image Classification

Download `train.zip` from the [Kaggle Dogs vs. Cats data page](https://www.kaggle.com/competitions/dogs-vs-cats/data), then run these cells in order. Upload `train.zip` when prompted. The notebook creates an 80/10/10 train/validation/test split, trains the MobileNetV2 CNN, reports metrics, and saves the graph and model.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import random
import shutil

uploaded = files.upload()  # Choose Kaggle's train.zip
zip_name = next(iter(uploaded))
raw_dir = Path('/content/dog_cat_raw')
data_dir = Path('/content/dog_cat_data')
raw_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as archive:
    archive.extractall(raw_dir)

random.seed(42)
for class_name in ['cat', 'dog']:
    images = [p for p in raw_dir.rglob(f'{class_name}.*') if p.is_file()]
    if not images:
        raise FileNotFoundError(f'No {class_name} images found. Upload Kaggle train.zip.')
    random.shuffle(images)
    train_end = int(0.80 * len(images))
    val_end = int(0.90 * len(images))
    partitions = {
        'train': images[:train_end],
        'validation': images[train_end:val_end],
        'test': images[val_end:],
    }
    for split, paths in partitions.items():
        target = data_dir / split / class_name
        target.mkdir(parents=True, exist_ok=True)
        for image_path in paths:
            shutil.copy2(image_path, target / image_path.name)

print('Dataset ready at:', data_dir)
for split in ['train', 'validation', 'test']:
    print(split, {name: len(list((data_dir / split / name).glob('*'))) for name in ['cat', 'dog']})

In [ ]:
!pip -q install tensorflow scikit-learn matplotlib numpy
!git clone https://github.com/Kishanth15/CNN-Based-Dog-vs.-Cat-Image-Classification.git /content/dog-cat-project
!python /content/dog-cat-project/cnn_dog_cat_classifier.py --data-dir /content/dog_cat_data --output-dir /content

In [ ]:
# Download the generated evaluation chart and trained model
from google.colab import files
files.download('/content/dog_cat_cnn_results.png')
files.download('/content/dog_cat_cnn.keras')